# Strategic asset allocation for a university endowment — walkthrough

**Mandate.** Build a long-only multi-asset portfolio that targets **≥ 7% nominal return p.a.** with **≤ 12% volatility**,
holds **≥ 8% cash**, **≥ 50% liquid** and **≥ 50% traditional** assets, and at least **three alternative asset classes** (≥ 5% each).
Internal limits: quarterly CVaR 95% ≥ −10% and inflation beta ≥ −1.

**Universe.** 9 asset classes (cash, US 10Y Treasuries, S&P 500, S&P Asia 50, direct & listed property, hedge funds,
commodities, private equity) + 6 ETFs/funds + Bitcoin, quarterly data 2006–Q1 2026.

**Approach.** Purely historical estimates over two windows (5Y: 2021–2025, 10Y: 2016–2025), so the result is transparent
and reproducible; current events are handled through scenario analysis rather than return forecasts.

In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
from portfolio_construction.data import load_levels, build_levels, quarterly_returns, window, ar1, geltner_desmooth, par_bond_duration_convexity, UNIVERSE
from portfolio_construction.stats import asset_statistics, portfolio_moments, risk_contributions
from portfolio_construction.optimize import Mandate, compliance_report, efficient_frontier, max_sharpe
from portfolio_construction.backtest import portfolio_returns, performance
pd.options.display.float_format = "{:,.4f}".format
ASSETS = list(UNIVERSE.index)
UNIVERSE

,label,traditional,liquid,class,max_weight,alternative
key,,,,,,
cash,Cash,1,1,Traditional,0.2000,0
bond,Bond (US 10Y govt),1,1,Traditional,0.4000,0
sp500,Equity - S&P 500,1,1,Traditional,0.4000,0
asia50,Equity - Asia 50,1,1,Traditional,0.2000,0
direct_property,Direct Property,0,0,Alt. A,0.1500,1
listed_property,Listed Property,0,1,Alt. B,0.1500,1
hedge_funds,Hedge Funds,0,0,Alt. C,0.1500,1
commodity,Commodity,0,1,Alt. D,0.1500,1
private_equity,Private Equity,0,0,Alt. E,0.1500,1


## 1. Data engineering
* **Cash**: 3-month T-bill rolled every quarter, $I_t = I_{t-1}(1 + y_{t-1}/400)$.
* **Bonds**: the bond index supplied had three implausible quarters, so a total-return series is rebuilt from the
  10Y Treasury yield with a duration–convexity approximation: $r_t \approx y_{t-1}/4 - D^{mod}\,\Delta y + \tfrac12 C\,\Delta y^2$.

In [2]:
levels = build_levels(load_levels("../data/quarterly_levels.csv"))
rets = quarterly_returns(levels)
print("Mod. duration and convexity of a 10Y par bond at 4%:", np.round(par_bond_duration_convexity(4.0), 3))
rets[["bond", "bond_index_original"]].loc["Q1 2016":].describe().loc[["mean", "std", "min", "max"]]

Mod. duration and convexity of a 10Y par bond at 4%: [ 8.176 78.898]


,bond,bond_index_original
mean,0.0029,0.0006
std,0.0403,0.0604
min,-0.0738,-0.2033
max,0.1212,0.1702


## 2. Risk and return of each asset class (10-year window)

In [3]:
w10 = window(rets, "Q1 2016", "Q4 2025"); w5 = window(rets, "Q1 2021", "Q4 2025")
s10 = asset_statistics(w10[ASSETS], w10["inflation"], w10["cash"])
s10.loc[["mean_ann", "cagr", "sd_ann", "sharpe", "cvar95_q", "max_drawdown", "inflation_beta", "ar1"]].T

,mean_ann,cagr,sd_ann,sharpe,cvar95_q,max_drawdown,inflation_beta,ar1
cash,0.0211,0.0213,0.0097,NaN,0.0001,0.0000,-0.1496,0.9692
bond,0.0123,0.0091,0.0816,-0.1084,-0.0717,-0.2410,-2.5548,-0.0108
sp500,0.1289,0.1265,0.1288,0.8366,-0.0910,-0.1592,-1.3117,-0.2058
asia50,0.1361,0.1217,0.2013,0.5711,-0.1742,-0.4394,-3.6102,0.0943
direct_property,0.0508,0.0508,0.0438,0.6778,-0.0275,-0.1360,1.4367,0.5815
listed_property,0.0331,0.0200,0.1657,0.0722,-0.1528,-0.3428,-0.2739,-0.1324
hedge_funds,0.0471,0.0465,0.0533,0.4875,-0.0569,-0.1109,-1.1802,0.4390
commodity,0.0726,0.0509,0.2157,0.2385,-0.2202,-0.4502,8.8825,-0.0609
private_equity,0.1661,0.1475,0.2235,0.6487,-0.2761,-0.3572,-1.6545,-0.1339
vaipx,0.0197,0.0185,0.0515,-0.0289,-0.0643,-0.1224,-0.5776,-0.1533


**Smoothing.** Direct property (appraisal-based) and hedge funds (reported NAVs) show strong positive autocorrelation,
so their measured volatility is artificially low. Geltner desmoothing $r^*_t = (r_t - \phi r_{t-1})/(1-\phi)$ roughly
doubles their volatility; it is used as a robustness check on portfolio risk.

In [4]:
for k in ["direct_property", "hedge_funds"]:
    phi = ar1(w10[k]); ds = window(geltner_desmooth(rets[k], phi), "Q1 2016", "Q4 2025")
    print(f"{k:<16} phi = {phi:.2f}   sd: {w10[k].std()*2:.2%} -> {ds.std()*2:.2%}")

direct_property  phi = 0.58   sd: 4.38% -> 8.49%
hedge_funds      phi = 0.44   sd: 5.33% -> 8.50%


## 3. The selected portfolio and the mandate checks

In [5]:
sel = pd.Series({"cash": .10, "bond": .05, "sp500": .35, "asia50": .05, "direct_property": .10,
                 "hedge_funds": .10, "commodity": .10, "private_equity": .10, "gdx": .05}).reindex(ASSETS).fillna(0)
mu5, mu10 = w5[ASSETS].mean() * 4, w10[ASSETS].mean() * 4
cov5, cov10 = w5[ASSETS].cov() * 4, w10[ASSETS].cov() * 4
compliance_report(sel, UNIVERSE, Mandate(), mu5, cov5, mu10, cov10)

,rule,portfolio,limit,status
0,Weights sum to 100%,1.0000,1.0000,PASS
1,No short selling,0.0000,0.0000,PASS
2,Within max weights,-0.0500,0.0000,PASS
3,Return >= 7% (5Y),0.1015,0.0700,PASS
4,Return >= 7% (10Y),0.1028,0.0700,PASS
5,Volatility <= 12% (5Y),0.0677,0.1200,PASS
6,Volatility <= 12% (10Y),0.0788,0.1200,PASS
7,Cash >= 8%,0.1000,0.0800,PASS
8,Liquid assets >= 50%,0.7000,0.5000,PASS
9,Traditional assets >= 50%,0.5500,0.5000,PASS


In [6]:
pd.DataFrame({"weight": sel, "risk share": risk_contributions(sel, cov10)}).query("weight > 0")

,weight,risk share
cash,0.1000,0.0014
bond,0.0500,0.0028
sp500,0.3500,0.5149
asia50,0.0500,0.1014
direct_property,0.1000,-0.0068
hedge_funds,0.1000,0.0348
commodity,0.1000,0.0929
private_equity,0.1000,0.2172
gdx,0.0500,0.0415


## 4. Back-test against a 60/40 benchmark

In [7]:
bench = pd.Series({"sp500": .6, "bond": .4}).reindex(ASSETS).fillna(0)
bt = pd.DataFrame({"Selected": portfolio_returns(rets[ASSETS], sel), "60/40": portfolio_returns(rets[ASSETS], bench)})
bt10 = window(bt, "Q1 2016", "Q4 2025")
pd.DataFrame({p: performance(bt10[p], w10["cash"], w10["inflation"]) for p in bt10})

,Selected,60/40
mean_ann,0.1028,0.0822
cagr,0.1036,0.0812
sd_ann,0.0788,0.0842
sharpe,1.0364,0.7256
worst_q,-0.0711,-0.0788
var95_q,-0.0296,-0.0596
cvar95_q,-0.0571,-0.0704
max_drawdown,-0.0967,-0.1610
skew,-0.2664,-0.5442
inflation_beta,-0.3214,-1.8089


## 5. Extension — what would an optimiser do?
The constrained efficient frontier (convex mandate constraints, max weights per asset) shows the selected portfolio
sits close to — but inside — the frontier. A max-Sharpe optimiser pushes into the assets with the best *past* returns
(gold miners, silver, Bitcoin at their caps): a textbook illustration of estimation error, and the reason the final
allocation was built with judgement on top of the numbers.

In [8]:
ef = efficient_frontier(UNIVERSE, Mandate(), mu10, cov10, 15)
ms = max_sharpe(UNIVERSE, Mandate(), mu10, cov10, mu10["cash"])
display(ef[["exp_return", "sd"]].T)
ms[ms > 0]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
exp_return,0.0355,0.0472,0.0590,0.0707,0.0824,0.0942,0.1059,0.1176,0.1294,0.1411,0.1528,0.1646,0.1763,0.1880,0.1998
sd,0.0272,0.0301,0.0351,0.0412,0.0478,0.0549,0.0622,0.0697,0.0773,0.0851,0.0931,0.1019,0.1123,0.1251,0.1585


key
cash              0.2000
sp500             0.3000
direct_property   0.1500
commodity         0.0490
gdx               0.1340
slv               0.0559
agthx             0.0611
btc               0.0500
dtype: float64